# 02 Full grid: Qwen2.5-7B-Instruct

**Purpose.** Run the full experiment grid for one model and export the traces.

- **Model:** `qwen2.5:7b-instruct-q4_K_M` (`configs/models.yaml` → `qwen25_7b`)
- **Grid** (`configs/grid.yaml`):
  - 24 tasks × 17 episodes (1 control + 4 attack styles × 4 delays) = 408 episodes.
  - Each episode is decoded at T=0 once and at T=0.7 with R=3 seeds (Mistral: T=0 only, `decoding_overrides`).
  - All 3 gates (none, scope_check, ticket_bound) are run by forking.
- **Expected runtime on a free T4:** about 3.5 h (1,632 runs at ~480 runs/h in pilot v3)
- **Prepared:** 2026-10-12 (after pilot v3)

**Resumable across sessions.** Traces are appended to `MyDrive/p2-netops/results/traces/qwen25_7b/traces.jsonl` after every run. When Colab disconnects or the daily limit ends, run all cells again later: finished runs are skipped.

In [1]:
# Setup 1: mount Drive, unpack the code, install Python deps
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/p2-netops'
import os
assert os.path.exists(f'{DRIVE}/p2-netops-code.zip'), 'Upload p2-netops-code.zip to MyDrive/p2-netops/ first'
!rm -rf /content/p2 && mkdir -p /content/p2 && unzip -oq {DRIVE}/p2-netops-code.zip -d /content/p2
!pip install -q -r /content/p2/requirements-colab.txt
import sys
sys.path.insert(0, '/content/p2/src')

ValueError: mount failed

In [ ]:
# Setup 2: install and start the Ollama server (inside Colab only)
!apt-get -qq install -y pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
from netinject.run import load_configs
GRID, MODELS = load_configs()   # all parameters come from configs/*.yaml
env = {**os.environ, 'OLLAMA_NUM_PARALLEL': str(GRID['parallel']),
       'OLLAMA_MAX_LOADED_MODELS': '1', 'OLLAMA_KEEP_ALIVE': '-1'}
server = subprocess.Popen(['ollama', 'serve'], env=env,
                          stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)

In [ ]:
# Environment record (goes into the manifest)
import platform, importlib.metadata as md
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
OLLAMA_VERSION = subprocess.run(['ollama', '--version'], capture_output=True, text=True).stdout.strip()
print('GPU:', GPU)
print('Ollama server:', OLLAMA_VERSION, '| ollama-python', md.version('ollama'),
      '| pydantic', md.version('pydantic'), '| python', platform.python_version())
assert GPU, 'No GPU found: set Runtime -> Change runtime type -> T4 GPU'
print('Grid:', {k: GRID.get(k) for k in ('tasks', 'decoding', 'decoding_overrides', 'max_steps', 'max_calls_per_turn', 'num_ctx', 'parallel')})

In [ ]:
# Full grid for qwen25_7b (resumable: finished runs are skipped; traces saved to Drive after every run)
import datetime, json, ollama
from pathlib import Path
from netinject.llm.ollama_client import OllamaClient
from netinject.run import plan, run_grid, write_manifest

MODEL_KEY = 'qwen25_7b'
tag = MODELS[MODEL_KEY]['tag']
decoding = GRID.get('decoding_overrides', {}).get(MODEL_KEY, GRID['decoding'])
jobs = plan(None if GRID['tasks'] == 'all' else GRID['tasks'], decoding)
print(MODEL_KEY, 'decoding:', decoding, '|', len(jobs), 'runs')
OUT = Path(DRIVE) / 'results' / 'traces' / MODEL_KEY
subprocess.run(['ollama', 'pull', tag], check=True, stdout=subprocess.DEVNULL)
digest = {m.model: m.digest for m in ollama.Client().list().models}[tag]
assert digest.startswith(MODELS[MODEL_KEY]['digest_prefix']), f'{tag}: digest {digest} differs from configs/models.yaml'
started = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
run_grid(OllamaClient(tag, GRID['num_ctx'], GRID['num_predict']), MODEL_KEY, OUT / 'traces.jsonl', jobs,
         GRID['max_steps'], GRID['parallel'], GRID['max_calls_per_turn'])
finished = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
m = write_manifest(OUT, f'02_{MODEL_KEY}_{started[:10]}', MODEL_KEY, tag, digest, OLLAMA_VERSION, GPU,
                   jobs, started, finished)
print(json.dumps({k: m[k] for k in ('episodes_expected', 'episodes_written', 'episodes_errored')}))

In [ ]:
# Completeness check + quick look (informational; official labels/metrics are recomputed locally)
import pandas as pd
from netinject.evaluation.judge import label_file
m = json.loads((OUT / 'manifest.json').read_text())
print('expected', m['episodes_expected'], '| written', m['episodes_written'], '| errored', m['episodes_errored'])
print('COMPLETE' if m['episodes_written'] == m['episodes_expected'] else 'INCOMPLETE: run all cells again to resume')
df = label_file(OUT / 'traces.jsonl')
att = df[df['style'] != 'control']
print(att.pivot_table(index='gate', columns='delay', values='unsafe_write_executed', aggfunc='mean').round(3))

In [ ]:
# Zip this model's traces for download
import shutil
zip_path = shutil.make_archive(f'{DRIVE}/results_{MODEL_KEY}', 'zip', root_dir=f'{DRIVE}/results',
                               base_dir=f'traces/{MODEL_KEY}')
print('Download', zip_path)

## After the run
1. File → Download → Download .ipynb, and replace `notebooks/colab/02_qwen25_7b.ipynb` with the executed copy.
2. Download `MyDrive/p2-netops/results_qwen25_7b.zip` and unzip it into the repo's `results/` folder.